<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%2010/Question_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Mixture Clustering as Conditional Updating

**A Complete Theoretical and Computational Treatment**

This notebook covers:
1. Derivation of the marginal density and posterior probabilities
2. One-hot encoding and soft assignments
3. Complete-data likelihood and the EM algorithm
4. Interactive visualization of GMM clustering on financial segmentation data

Run all cells sequentially in Google Colab.

In [1]:
# ============================================================
# CELL 1: INSTALLATION & IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import multivariate_normal
import warnings
warnings.filterwarnings('ignore')

print("All packages imported successfully!")

All packages imported successfully!


## Part 1: Deriving the Marginal Density

### Derivation

We begin with the law of total probability. Since the latent cluster indicator $C_i$ forms a partition of the probability space (exactly one cluster must be chosen), we can marginalize over all possible values of $C_i$:

$$p(x_i) = \sum_{k=1}^{K} p(x_i \mid C_i = k) \cdot P(C_i = k)$$

By the model specification:
- $P(C_i = k) = \phi_k$ (the mixture weight / prior probability of cluster $k$)
- $p(x_i \mid C_i = k) = \mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ (the multivariate Gaussian density)

Substituting these gives:

$$\boxed{p(x_i) = \sum_{k=1}^{K} \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}$$

### Why is this called a Gaussian Mixture Density?

The marginal density $p(x_i)$ is called a **Gaussian mixture density** because it is a **convex combination** (weighted sum) of $K$ individual Gaussian (normal) densities. Each component $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ is a Gaussian "bump" centered at $\mu_k$ with shape $\Sigma_k$, and the weights $\phi_k$ ensure the total probability mass integrates to 1.

Unlike a single Gaussian which is unimodal, a mixture of Gaussians can model **multimodal, skewed, or complex-shaped distributions** by superimposing multiple Gaussian components. The flexibility comes from the fact that any smooth density can be approximated arbitrarily well by a mixture of enough Gaussians.

## Part 2: Deriving the Posterior Cluster Probability (Responsibility)

### Derivation via Bayes' Rule

For a fixed observation $x_i$, we want the probability that it came from cluster $k$ **after** having seen the data. By Bayes' theorem:

$$P(C_i = k \mid X_i = x_i) = \frac{P(X_i = x_i \mid C_i = k) \cdot P(C_i = k)}{P(X_i = x_i)}$$

The denominator is the marginal likelihood derived in Part 1:

$$P(X_i = x_i) = \sum_{j=1}^{K} P(X_i = x_i \mid C_i = j) \cdot P(C_i = j)$$

Substituting the Gaussian likelihood and the prior $\phi_k$:

$$\boxed{\gamma_{ik} = P(C_i = k \mid X_i = x_i) = \frac{\phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \phi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)}}$$

### Why is $\gamma_{ik}$ called a posterior probability?

- **Prior**: Before seeing $x_i$, the probability that point $i$ belongs to cluster $k$ is simply $\phi_k$. This is our "belief" based only on the model structure.
- **Likelihood**: $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ measures how probable the observed data $x_i$ is under cluster $k$.
- **Posterior**: After observing $x_i$, we update our belief using Bayes' rule. The quantity $\gamma_{ik}$ represents the **updated probability** of cluster membership given the evidence.

Because $\gamma_{ik}$ is computed *after* observing the data, it is a **posterior probability**. It is called the **responsibility** because it quantifies how much cluster $k$ is "responsible for" or "accountable for" generating data point $x_i$.

## Part 3: One-Hot Encoding of the Latent Cluster Variable

### Setup

Define the one-hot encoded vector $Z_i \in \{0,1\}^K$ where:

$$Z_{ik} = \begin{cases} 1 & \text{if } C_i = k \\ 0 & \text{otherwise} \end{cases}$$

### Derivation of the Conditional Expectation

We show that $\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i)$.

Since $Z_{ik}$ is an indicator variable (takes only values 0 or 1), its conditional expectation is:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = 1 \cdot P(Z_{ik} = 1 \mid X_i = x_i) + 0 \cdot P(Z_{ik} = 0 \mid X_i = x_i)$$

But $Z_{ik} = 1$ if and only if $C_i = k$, so:

$$\mathbb{E}[Z_{ik} \mid X_i = x_i] = P(C_i = k \mid X_i = x_i) = \gamma_{ik}$$

### Vector Form

Stacking all $K$ components:

$$\mathbb{E}[Z_i \mid X_i = x_i] = \begin{bmatrix} \mathbb{E}[Z_{i1} \mid X_i = x_i] \\ \mathbb{E}[Z_{i2} \mid X_i = x_i] \\ \vdots \\ \mathbb{E}[Z_{iK} \mid X_i = x_i] \end{bmatrix} = \begin{bmatrix} \gamma_{i1} \\ \gamma_{i2} \\ \vdots \\ \gamma_{iK} \end{bmatrix}$$

### Conclusion: Soft Assignment as Conditional Expectation

The soft cluster assignment in a Gaussian Mixture Model is precisely:

$$\boxed{\mathbb{E}[Z_i \mid X_i = x_i]}$$

This is a beautiful result: instead of making a hard binary decision about which cluster a point belongs to, the GMM provides a **probability vector** where each entry $\gamma_{ik}$ tells us the *degree* to which point $i$ belongs to cluster $k$. The assignment is "soft" because a single data point can partially belong to multiple clusters simultaneously, with the partial memberships summing to 1.

## Part 4: From Soft Assignment to Hard Clustering

### Hard Cluster Assignment

From the soft assignment vector $\mathbb{E}[Z_i \mid X_i = x_i] = (\gamma_{i1}, \ldots, \gamma_{iK})^T$, we obtain a hard assignment by selecting the cluster with maximum posterior probability:

$$\widehat{C}_i = \arg\max_{1 \leq k \leq K} \gamma_{ik}$$

### Difference Between Soft and Hard Clustering

| Aspect | Soft Clustering | Hard Clustering |
|--------|----------------|-----------------|
| **Output** | Probability vector $\gamma_i = (\gamma_{i1}, \ldots, \gamma_{iK})$ | Single integer label $\widehat{C}_i \in \{1, \ldots, K\}$ |
| **Information** | Retains uncertainty; shows ambiguity | Discards uncertainty |
| **Membership** | Point $i$ can belong 70% to cluster 1 and 30% to cluster 2 | Point $i$ belongs exclusively to cluster 1 |
| **Use Case** | Density estimation, understanding cluster overlap, probabilistic decision making | Classification, creating distinct segments |
| **Computation** | Natural output of GMM's E-step | Requires an additional $\arg\max$ step |

### Key Insight

Soft clustering respects the **uncertainty inherent in the data**. If a point lies exactly between two clusters, hard clustering forces an arbitrary binary choice, while soft clustering quantifies the ambiguity. In financial segmentation (e.g., customer clustering), soft assignments can reveal customers who exhibit mixed behaviors and don't fit neatly into a single category.

## Part 5: Conditional Expectation of the Observation Given the Cluster

### Derivation

Given that $X_i \mid C_i = k \sim \mathcal{N}(\mu_k, \Sigma_k)$, by the definition of the multivariate Gaussian distribution:

$$\boxed{\mathbb{E}[X_i \mid C_i = k] = \mu_k}$$

### Interpretation of $\mu_k$ as the Cluster Center

The mean vector $\mu_k$ represents the **centroid** or **center of gravity** of cluster $k$. All points generated from cluster $k$ are distributed around $\mu_k$ according to the covariance $\Sigma_k$. Geometrically, $\mu_k$ is the point in $\mathbb{R}^d$ where the Gaussian density $\mathcal{N}(x \mid \mu_k, \Sigma_k)$ attains its maximum value.

### Comparing the Two Conditional Expectations

| Conditional Expectation | Direction | Meaning |
|------------------------|-----------|---------|
| $\mathbb{E}[Z_i \mid X_i = x_i]$ | Data $\rightarrow$ Cluster | Given an **observed point** $x_i$, what is the probability distribution over clusters? This gives the **soft membership** of the point. |
| $\mathbb{E}[X_i \mid C_i = k]$ | Cluster $\rightarrow$ Data | Given a **cluster** $k$, where do we expect points to be located? This gives the **mean location** of the cluster. |

### Why the Distinction Matters

- **$\mathbb{E}[Z_i \mid X_i = x_i]$** answers: *"I see a data point; which cluster does it most likely belong to?"* (inference)
- **$\mathbb{E}[X_i \mid C_i = k]$** answers: *"I know the cluster; where should I expect to find its points?"* (generative modeling)

This duality is the heart of the EM algorithm: we alternate between inferring cluster memberships (E-step, using $\mathbb{E}[Z_i \mid X_i]$) and updating the cluster locations (M-step, using $\mathbb{E}[X_i \mid C_i = k]$).

## Part 6: The Complete-Data Likelihood

### Derivation of the Log-Likelihood

If the latent labels $z_i$ were observed, the joint probability factors because observations are independent given the cluster assignments:

$$p(x_1, \ldots, x_n, z_1, \ldots, z_n) = \prod_{i=1}^{n} p(x_i, z_i) = \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}}$$

The exponent $z_{ik}$ acts as a selector: when $z_{ik} = 1$, only the $k$-th term contributes; when $z_{ik} = 0$, the term becomes 1 and disappears from the product.

Taking the natural logarithm:

$$\ell_c = \log \left( \prod_{i=1}^{n} \prod_{k=1}^{K} \left[ \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]^{z_{ik}} \right)$$

$$\ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \log \left( \phi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right)$$

$$\boxed{\ell_c = \sum_{i=1}^{n} \sum_{k=1}^{K} z_{ik} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]}$$

### Why is this easy to maximize if $z_{ik}$ were known?

If $z_{ik}$ were observed, the complete-data log-likelihood **decouples** into separate terms for each cluster:

- The $\phi_k$ terms involve only the points assigned to cluster $k$
- The $\mu_k$ and $\Sigma_k$ terms are just the log-likelihood of a **single Gaussian** fit to the points in cluster $k$

Specifically:
- $\phi_k$ is maximized by the empirical proportion of points in cluster $k$
- $\mu_k$ is the sample mean of points in cluster $k$
- $\Sigma_k$ is the sample covariance of points in cluster $k$

These have **closed-form solutions** because the constraints and derivatives are straightforward when the cluster memberships are known. The difficulty in GMM arises precisely because $z_{ik}$ is **latent** (unobserved).

## Part 7: The EM Interpretation

### The E-Step: Replacing Latent Variables with Their Conditional Expectations

Since $z_{ik}$ is unobserved, the EM algorithm replaces it with its conditional expectation given the observed data and current parameter estimates $\theta^{(t)} = (\phi^{(t)}, \mu^{(t)}, \Sigma^{(t)})$:

$$z_{ik} \quad \leadsto \quad \mathbb{E}[Z_{ik} \mid X_i = x_i, \theta^{(t)}] = \gamma_{ik}^{(t)}$$

This substitution yields the **expected complete-data log-likelihood** (the Q-function):

$$\boxed{Q(\theta \mid \theta^{(t)}) = \sum_{i=1}^{n} \sum_{k=1}^{K} \gamma_{ik}^{(t)} \left[ \log \phi_k + \log \mathcal{N}(x_i \mid \mu_k, \Sigma_k) \right]}$$

### Why is the E-step a conditional update of cluster membership?

The E-step computes $\gamma_{ik}^{(t)}$ using the **current** parameter estimates. This is a **conditional update** because:

1. It conditions on the observed data $X_i = x_i$
2. It uses the current belief about the parameters $\theta^{(t)}$
3. It outputs the **posterior probability** $P(C_i = k \mid X_i = x_i, \theta^{(t)})$

This updates our knowledge about the latent cluster memberships from the prior $\phi_k$ to the posterior $\gamma_{ik}$. The term "conditional" reflects that we are computing expectations **conditioned on** the observed data.

### The EM Cycle

The EM algorithm iterates:
- **E-step**: Compute $\gamma_{ik}^{(t)} = \mathbb{E}[Z_{ik} \mid x_i, \theta^{(t)}]$ (conditional expectation)
- **M-step**: Maximize $Q(\theta \mid \theta^{(t)})$ to obtain $\theta^{(t+1)}$

Each iteration is guaranteed to increase (or leave unchanged) the observed-data log-likelihood.

## Part 8: Parameter Updates (The M-Step)

### Derivation of the Updates

We maximize $Q$ with respect to $\phi_k, \mu_k, \Sigma_k$ subject to $\sum_{k=1}^{K} \phi_k = 1$.

#### 1. Effective Cluster Size

Define the **effective number of points** assigned to cluster $k$:

$$\boxed{N_k = \sum_{i=1}^{n} \gamma_{ik}}$$

Unlike hard clustering where $N_k$ counts points, here $N_k$ sums **fractional memberships**. A point that is 60% in cluster 1 and 40% in cluster 2 contributes 0.6 to $N_1$ and 0.4 to $N_2$.

#### 2. Mixture Weights

Using a Lagrange multiplier for the constraint $\sum_k \phi_k = 1$:

$$\frac{\partial Q}{\partial \phi_k} = \sum_{i=1}^{n} \frac{\gamma_{ik}}{\phi_k} - \lambda = 0 \quad \Rightarrow \quad \boxed{\phi_k^{\text{new}} = \frac{N_k}{n}}$$

#### 3. Mean Vectors

$$\frac{\partial Q}{\partial \mu_k} = \sum_{i=1}^{n} \gamma_{ik} \Sigma_k^{-1}(x_i - \mu_k) = 0 \quad \Rightarrow \quad \boxed{\mu_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik} x_i}$$

This is a **weighted average** of all data points, where each point $x_i$ is weighted by its responsibility $\gamma_{ik}$.

#### 4. Covariance Matrices

$$\frac{\partial Q}{\partial \Sigma_k} = 0 \quad \Rightarrow \quad \boxed{\Sigma_k^{\text{new}} = \frac{1}{N_k} \sum_{i=1}^{n} \gamma_{ik} (x_i - \mu_k^{\text{new}})(x_i - \mu_k^{\text{new}})^T}$$

### Role of $\gamma_{ik}$ as Fractional Membership Weight

The responsibility $\gamma_{ik}$ acts as a **soft label** or **fractional weight**:

- In hard K-means, a point either belongs to cluster $k$ (weight = 1) or it doesn't (weight = 0).
- In GMM, every point contributes to **every cluster** proportionally to $\gamma_{ik}$.
- A point near the boundary between two clusters will have similar $\gamma$ values for both, and thus will pull both cluster centers toward it.
- A point deep inside cluster $k$ will have $\gamma_{ik} \approx 1$ and will dominantly influence $\mu_k$ and $\Sigma_k$.

This fractional weighting is what makes GMM robust and probabilistically principled.

## Part 9: Interpretation — GMM as Repeated Conditional Updating

Gaussian Mixture clustering can be understood as a repeated process of **conditional updating** that alternates between inferring hidden structure and refining model parameters.

The mixture weight $\phi_k$ represents the **prior probability** of cluster $k$—our initial belief about how prevalent the cluster is in the population before seeing any data. The Gaussian density $\mathcal{N}(x_i \mid \mu_k, \Sigma_k)$ measures how **compatible** the observation $x_i$ is with cluster $k$, quantifying the likelihood of observing $x_i$ if it truly belonged to that cluster. When we apply Bayes' rule, these two pieces of information combine to produce the **responsibility** $\gamma_{ik}$, which is the **posterior probability** of cluster membership after observing $x_i$. This posterior updates our beliefs: clusters that are both probable a priori and consistent with the data receive high responsibility values.

The soft assignment vector $\mathbb{E}[Z_i \mid X_i = x_i] = (\gamma_{i1}, \ldots, \gamma_{iK})^T$ encodes the full posterior distribution over clusters for each data point. Rather than forcing a binary decision, it preserves uncertainty by distributing probability mass across all clusters. In the M-step, these posterior probabilities serve as **weights** for updating the cluster parameters: the new mean $\mu_k^{\text{new}}$ is a weighted average of all points, the new covariance $\Sigma_k^{\text{new}}$ is a weighted sample covariance, and the new mixture weight $\phi_k^{\text{new}}$ is the normalized total responsibility.

Thus, Gaussian mixture clustering is fundamentally **probabilistic clustering based on conditional expectations of latent cluster membership variables**. Each EM iteration is a cycle of conditioning: we condition on the data to estimate the latent variables (E-step), then condition on the latent variables to estimate the parameters (M-step). This elegant interplay between observed and hidden quantities is what gives the GMM its power and interpretability.

## Part 10: Computational Simulation and Out-of-Sample Validation

### Dataset Information

We use the **Credit Card Customer Dataset** (CC_GENERAL.csv) from Kaggle, which summarizes usage behavior of ~9000 active credit card holders.

**Key features used:**
- `PURCHASES`: Amount of purchases made from account
- `CREDIT_LIMIT`: Limit of Credit Card for user

**Note on Data Access:**
Since Kaggle requires authentication, you have two options:
1. **Manual Upload**: Download `CC_GENERAL.csv` from [Kaggle](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata) and upload it using the file panel on the left in Colab.
2. **Synthetic Data**: Run the cell below to generate synthetic data with similar statistical properties.

In [2]:
# ============================================================
# CELL 2: DATA LOADING (with synthetic fallback)
# ============================================================
import os

# Try to load the real dataset; if not found, generate synthetic data
DATA_PATH = '/content/CC_GENERAL.csv'  # Upload your file here in Colab

def generate_synthetic_cc_data(n_samples=9000, random_state=42):
    np.random.seed(random_state)

    # Cluster 0: Low spenders, low credit limit
    n0 = int(0.35 * n_samples)
    mean0 = [800, 3000]
    cov0 = [[40000, 20000], [20000, 250000]]
    X0 = np.random.multivariate_normal(mean0, cov0, n0)

    # Cluster 1: Moderate spenders, medium credit limit
    n1 = int(0.40 * n_samples)
    mean1 = [2500, 7500]
    cov1 = [[250000, 80000], [80000, 900000]]
    X1 = np.random.multivariate_normal(mean1, cov1, n1)

    # Cluster 2: High spenders, high credit limit
    n2 = n_samples - n0 - n1
    mean2 = [6000, 15000]
    cov2 = [[900000, 200000], [200000, 2500000]]
    X2 = np.random.multivariate_normal(mean2, cov2, n2)

    X = np.vstack([X0, X1, X2])
    X = np.abs(X)

    df = pd.DataFrame(X, columns=['PURCHASES', 'CREDIT_LIMIT'])
    df['PURCHASES'] = df['PURCHASES'].clip(lower=0)
    df['CREDIT_LIMIT'] = df['CREDIT_LIMIT'].clip(lower=500)

    return df

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    df = df[['PURCHASES', 'CREDIT_LIMIT']].dropna()
    print("Loaded real CC_GENERAL.csv dataset")
else:
    print("CC_GENERAL.csv not found. Generating synthetic data...")
    print("To use real data: upload CC_GENERAL.csv to /content/ in Colab")
    df = generate_synthetic_cc_data(n_samples=9000)
    print("Synthetic data generated with 3 clusters")

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nDescriptive statistics:")
print(df.describe())

CC_GENERAL.csv not found. Generating synthetic data...
To use real data: upload CC_GENERAL.csv to /content/ in Colab
Synthetic data generated with 3 clusters
Dataset shape: (9000, 2)

First 5 rows:
     PURCHASES  CREDIT_LIMIT
0   796.553543   3250.726378
1  1126.562401   3295.685436
2   743.449858   2887.297510
3  1023.638733   3774.995742
4   883.309921   2755.467319

Descriptive statistics:
          PURCHASES  CREDIT_LIMIT
count   9000.000000   9000.000000
mean    2788.202853   7803.745935
std     2093.879073   4720.359297
min      130.920892   1399.250497
25%      909.560150   3271.730622
50%     2352.366569   7177.975040
75%     3714.574778  10294.767164
max    10029.539994  20102.119998


In [3]:
# ============================================================
# CELL 3: GMMFinancialSegmenter Implementation
# ============================================================

class GMMFinancialSegmenter:
    """
    A Gaussian Mixture Model segmenter for 2D financial data.
    Implements data splitting, scaling, EM fitting, and interactive Plotly visualizations.
    """

    def __init__(self, n_components=3, random_state=42, test_size=0.2):
        self.n_components = n_components
        self.random_state = random_state
        self.test_size = test_size
        self.scaler = StandardScaler()
        self.gmm = None
        self.X_train = None
        self.X_test = None
        self.X_train_scaled = None
        self.X_test_scaled = None
        self.features = None
        self.converged = None
        self.n_iter = None
        self.responsibilities_train = None
        self.responsibilities_test = None

    def fit(self, df, feature_cols=None):
        if feature_cols is None:
            feature_cols = ['PURCHASES', 'CREDIT_LIMIT']
        self.features = feature_cols
        X = df[feature_cols].values
        self.X_train, self.X_test = train_test_split(
            X, test_size=self.test_size, random_state=self.random_state
        )
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)
        self.gmm = GaussianMixture(
            n_components=self.n_components,
            covariance_type='full',
            random_state=self.random_state,
            max_iter=200,
            n_init=10,
            init_params='kmeans'
        )
        self.gmm.fit(self.X_train_scaled)
        self.converged = self.gmm.converged_
        self.n_iter = self.gmm.n_iter_
        self.responsibilities_train = self.gmm.predict_proba(self.X_train_scaled)
        self.responsibilities_test = self.gmm.predict_proba(self.X_test_scaled)
        print("=" * 60)
        print("GMM FITTING RESULTS")
        print("=" * 60)
        print(f"Converged: {self.converged}")
        print(f"Iterations: {self.n_iter}")
        print(f"Training samples: {len(self.X_train)}")
        print(f"Test samples: {len(self.X_test)}")
        print(f"Log-likelihood (train): {self.gmm.score(self.X_train_scaled):.4f}")
        print(f"AIC: {self.gmm.aic(self.X_train_scaled):.2f}")
        print(f"BIC: {self.gmm.bic(self.X_train_scaled):.2f}")
        print("=" * 60)
        means_orig = self.scaler.inverse_transform(self.gmm.means_)
        print("\nLearned Cluster Centers (original scale):")
        for k in range(self.n_components):
            print(f"  Cluster {k}: PURCHASES={means_orig[k][0]:.2f}, CREDIT_LIMIT={means_orig[k][1]:.2f}")
        print(f"  Mixture weights: {self.gmm.weights_}")
        print("=" * 60)
        return self

    def test_log_likelihood(self):
        avg_ll = self.gmm.score(self.X_test_scaled)
        print(f"\nOut-of-Sample Performance:")
        print(f"Average log-likelihood on test set: {avg_ll:.4f}")
        return avg_ll

    def _create_grid(self, n_points=200):
        x_min = self.X_train[:, 0].min() * 0.9
        x_max = self.X_train[:, 0].max() * 1.1
        y_min = self.X_train[:, 1].min() * 0.9
        y_max = self.X_train[:, 1].max() * 1.1
        xx, yy = np.meshgrid(
            np.linspace(x_min, x_max, n_points),
            np.linspace(y_min, y_max, n_points)
        )
        return xx, yy, x_min, x_max, y_min, y_max

    def plot_2d_density(self):
        fig = make_subplots(
            rows=2, cols=2,
            specs=[[{"type": "xy"}, {"type": "xy"}],
                   [{"type": "xy", "colspan": 2}, None]],
            subplot_titles=("Marginal: PURCHASES", "Marginal: CREDIT_LIMIT",
                           "2D Density Heatmap (Training Data)"),
            row_heights=[0.25, 0.75],
            column_widths=[0.5, 0.5]
        )
        fig.add_trace(
            go.Histogram(x=self.X_train[:, 0], nbinsx=50,
                        marker_color='steelblue', opacity=0.7,
                        name='PURCHASES dist'),
            row=1, col=1
        )
        fig.add_trace(
            go.Histogram(x=self.X_train[:, 1], nbinsx=50,
                        marker_color='coral', opacity=0.7,
                        name='CREDIT_LIMIT dist'),
            row=1, col=2
        )
        fig.add_trace(
            go.Histogram2dContour(
                x=self.X_train[:, 0], y=self.X_train[:, 1],
                colorscale='Viridis', ncontours=30,
                contours=dict(coloring='heatmap'),
                colorbar=dict(title='Density', x=1.02),
                name='Density'
            ),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=self.X_train[:, 0], y=self.X_train[:, 1],
                mode='markers', marker=dict(size=3, color='white', opacity=0.3),
                name='Data points'
            ),
            row=2, col=1
        )
        fig.update_xaxes(title_text="PURCHASES ($)", row=2, col=1)
        fig.update_yaxes(title_text="CREDIT_LIMIT ($)", row=2, col=1)
        fig.update_layout(height=800, width=900, showlegend=False,
                         title_text="Empirical 2D Density with Marginal Distributions")
        return fig

    def plot_training_assignment(self, n_grid=150):
        xx, yy, x_min, x_max, y_min, y_max = self._create_grid(n_grid)
        grid_points = np.c_[xx.ravel(), yy.ravel()]
        grid_scaled = self.scaler.transform(grid_points)
        resp_grid = self.gmm.predict_proba(grid_scaled)
        max_resp_grid = resp_grid.max(axis=1).reshape(xx.shape)
        train_labels = self.responsibilities_train.argmax(axis=1)
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        fig = go.Figure()
        fig.add_trace(
            go.Contour(
                z=max_resp_grid, x=xx[0], y=yy[:, 0],
                colorscale='RdYlBu', contours=dict(start=0, end=1, size=0.05),
                colorbar=dict(title='max(γ_ik)', x=1.02),
                name='Max Responsibility',
                opacity=0.6
            )
        )
        for k in range(self.n_components):
            mask = train_labels == k
            fig.add_trace(
                go.Scatter(
                    x=self.X_train[mask, 0], y=self.X_train[mask, 1],
                    mode='markers',
                    marker=dict(size=6, color=colors[k],
                               line=dict(width=1, color='black')),
                    name=f'Train Cluster {k}'
                )
            )
        centers_orig = self.scaler.inverse_transform(self.gmm.means_)
        fig.add_trace(
            go.Scatter(
                x=centers_orig[:, 0], y=centers_orig[:, 1],
                mode='markers+text',
                marker=dict(size=15, color='gold', symbol='star',
                           line=dict(width=2, color='black')),
                text=[f'μ_{k}' for k in range(self.n_components)],
                textposition='top center',
                name='Cluster Centers'
            )
        )
        fig.update_xaxes(title_text="PURCHASES ($)", range=[x_min, x_max])
        fig.update_yaxes(title_text="CREDIT_LIMIT ($)", range=[y_min, y_max])
        fig.update_layout(
            title="Training Assignment: Soft Responsibility Contours with Hard Labels",
            height=650, width=800
        )
        return fig

    def plot_test_assignment(self, n_grid=150):
        xx, yy, x_min, x_max, y_min, y_max = self._create_grid(n_grid)
        grid_points = np.c_[xx.ravel(), yy.ravel()]
        grid_scaled = self.scaler.transform(grid_points)
        resp_grid = self.gmm.predict_proba(grid_scaled)
        max_resp_grid = resp_grid.max(axis=1).reshape(xx.shape)
        test_labels = self.responsibilities_test.argmax(axis=1)
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        fig = go.Figure()
        fig.add_trace(
            go.Contour(
                z=max_resp_grid, x=xx[0], y=yy[:, 0],
                colorscale='RdYlBu', contours=dict(start=0, end=1, size=0.05),
                colorbar=dict(title='max(γ_ik)', x=1.02),
                name='Max Responsibility',
                opacity=0.5
            )
        )
        for k in range(self.n_components):
            mask = test_labels == k
            if mask.sum() > 0:
                fig.add_trace(
                    go.Scatter(
                        x=self.X_test[mask, 0], y=self.X_test[mask, 1],
                        mode='markers',
                        marker=dict(size=10, color=colors[k], symbol='diamond',
                                   line=dict(width=2, color='black'),
                                   opacity=0.9),
                        name=f'Test Cluster {k}'
                    )
                )
        centers_orig = self.scaler.inverse_transform(self.gmm.means_)
        fig.add_trace(
            go.Scatter(
                x=centers_orig[:, 0], y=centers_orig[:, 1],
                mode='markers+text',
                marker=dict(size=15, color='gold', symbol='star',
                           line=dict(width=2, color='black')),
                text=[f'μ_{k}' for k in range(self.n_components)],
                textposition='top center',
                name='Cluster Centers'
            )
        )
        fig.update_xaxes(title_text="PURCHASES ($)", range=[x_min, x_max])
        fig.update_yaxes(title_text="CREDIT_LIMIT ($)", range=[y_min, y_max])
        fig.update_layout(
            title="Test Assignment: Out-of-Sample Points on Responsibility Contours",
            height=650, width=800
        )
        return fig

print("GMMFinancialSegmenter class defined successfully!")

GMMFinancialSegmenter class defined successfully!


In [4]:
# ============================================================
# CELL 4: FIT THE MODEL
# ============================================================
segmenter = GMMFinancialSegmenter(n_components=3, random_state=42, test_size=0.2)
segmenter.fit(df)

GMM FITTING RESULTS
Converged: True
Iterations: 2
Training samples: 7200
Test samples: 1800
Log-likelihood (train): -0.6054
AIC: 8752.43
BIC: 8869.42

Learned Cluster Centers (original scale):
  Cluster 0: PURCHASES=798.95, CREDIT_LIMIT=3005.98
  Cluster 1: PURCHASES=6038.20, CREDIT_LIMIT=15051.18
  Cluster 2: PURCHASES=2501.81, CREDIT_LIMIT=7492.69
  Mixture weights: [0.35042894 0.24933931 0.40023175]


In [5]:
# ============================================================
# CELL 5: OUT-OF-SAMPLE PERFORMANCE
# ============================================================
test_ll = segmenter.test_log_likelihood()


Out-of-Sample Performance:
Average log-likelihood on test set: -0.6415


In [6]:
# ============================================================
# CELL 6: VISUALIZATION 1 - 2D Density Heatmap
# ============================================================
fig_density = segmenter.plot_2d_density()
fig_density.show()

In [7]:
# ============================================================
# CELL 7: VISUALIZATION 2 - Training Assignment Plot
# ============================================================
fig_train = segmenter.plot_training_assignment(n_grid=150)
fig_train.show()

In [8]:
# ============================================================
# CELL 8: VISUALIZATION 3 - Test Assignment Plot
# ============================================================
fig_test = segmenter.plot_test_assignment(n_grid=150)
fig_test.show()

## Evaluation and Interpretation of Part 10 Results

### What the Visualizations Show

#### 1. Empirical 2D Density Heatmap
The first figure reveals the **multimodal structure** of the financial data. The marginal histograms show right-skewed distributions typical of financial variables (most customers have low-to-moderate purchases and credit limits, with a long tail of high-value customers). The 2D density contour shows whether the data naturally forms separated peaks or elongated shapes, justifying the use of a mixture model over a single Gaussian.

#### 2. Training Assignment Plot
This is the most theoretically important figure. The **continuous background contour** encodes $\max_k \gamma_{ik}(x_{\text{grid}})$—the highest posterior responsibility at each grid location.

This background is a direct visualization of the **soft assignment expectation vector** $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that we proved analytically in Part 3. Specifically:
- At any grid point $x_{\text{grid}}$, the GMM computes the full responsibility vector $(\gamma_1, \gamma_2, \gamma_3)$.
- The contour color represents $\max_k \gamma_k$, which tells us how "confident" the model is about the assignment at that location.
- **Regions where the contour is dark red** (high value near 1) are deep inside a cluster—here, one $\gamma_k \approx 1$ and the others are near 0. The model is certain.
- **Regions where the contour transitions through yellow/green** (values around 0.5–0.7) are **ambiguity zones** where two or more clusters share responsibility. These are the boundaries where soft assignment matters most.

The overlaid training points, colored by their hard $\arg\max$ label, show how the empirical data populate these regions. Points lying in ambiguous transition zones are precisely those for which the soft assignment vector $\mathbb{E}[Z_i \mid X_i = x_i]$ spreads probability mass across multiple clusters.

#### 3. Test Assignment Plot
By overlaying **out-of-sample** test points on the same responsibility contours, we validate whether the learned density generalizes. If test points fall into regions where $\max_k \gamma_k$ is high, the model generalizes confidently. If many test points land in low-confidence transition zones, this exposes **cluster ambiguity** in the population and suggests either overlapping clusters or a need for more components.

### Connection to Theory

The continuous contour map is the computational realization of the analytical derivation in Part 3. While Part 3 proved that $\mathbb{E}[Z_i \mid X_i = x_i] = (\gamma_{i1}, \ldots, \gamma_{iK})^T$ for discrete data points, the grid evaluation extends this to the **entire feature space**. Every pixel in the background contour corresponds to computing the conditional expectation of the one-hot latent vector at that coordinate.

This demonstrates that GMM clustering is not just about assigning labels—it is about **learning a continuous probability field** over the input space. The EM algorithm iteratively updates this field (E-step) and the Gaussian parameters that define it (M-step), which is exactly the "repeated process of conditional updating" described in Part 9.

### Why This Matters for Financial Segmentation

In credit card customer segmentation, hard clustering might force a customer with moderate purchases and a medium credit limit into a single segment. The soft assignment contour reveals that such a customer might actually lie in a region where the posterior probabilities are split between, say, a "standard" cluster and a "premium" cluster. This uncertainty is valuable business information: it identifies customers who are **on the boundary** between segments and might be targeted for upselling or special offers.

### Conclusion

This notebook has shown both analytically and computationally that Gaussian Mixture clustering is a principled probabilistic framework. The soft assignments are conditional expectations of latent variables, the EM algorithm is a repeated conditional updating procedure, and the resulting responsibility contours provide an interpretable, continuous map of cluster membership uncertainty across the entire feature space.

Using the given dataset, re-evaluation

In [9]:
# ============================================================
# PART 10: COMPUTATIONAL SIMULATION & OUT-OF-SAMPLE VALIDATION
# ============================================================
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------
# 1. DATA LOADING (KaggleHub + synthetic fallback)
# ------------------------------------------------------------------
def load_credit_data():
    """Load CC_GENERAL.csv via kagglehub, or generate synthetic data."""
    try:
        import kagglehub
        path = kagglehub.dataset_download("arjunbhasin2013/ccdata")
        csv_path = os.path.join(path, "CC GENERAL.csv")
        df = pd.read_csv(csv_path)
        df = df[["PURCHASES", "CREDIT_LIMIT"]].dropna()
        print(f"Loaded real Kaggle data: {df.shape[0]} rows")
        return df
    except Exception as e:
        print(f"Kaggle load failed ({e}). Using synthetic data...")
        np.random.seed(42)
        n = 9000
        # Three financial segments: low / mid / premium
        c1 = np.random.multivariate_normal([800, 3000],  [[40000,  20000], [20000,  250000]], int(0.35*n))
        c2 = np.random.multivariate_normal([2500, 7500], [[250000, 80000], [80000,  900000]], int(0.40*n))
        c3 = np.random.multivariate_normal([6000, 15000],[[900000,200000], [200000,2500000]], n - int(0.35*n) - int(0.40*n))
        X = np.vstack([c1, c2, c3])
        X = np.abs(X)
        df = pd.DataFrame(X, columns=["PURCHASES", "CREDIT_LIMIT"])
        df = df.clip(lower={"PURCHASES": 0, "CREDIT_LIMIT": 500})
        print(f"Generated synthetic data: {df.shape[0]} rows")
        return df


# ------------------------------------------------------------------
# 2. GMM SEGMENTER CLASS
# ------------------------------------------------------------------
class GMMFinancialSegmenter:
    """
    End-to-end GMM pipeline for 2D financial segmentation.
    Fits K=3 components, reports EM convergence, and renders
    three interactive Plotly figures.
    """

    def __init__(self, n_components=3, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.model = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            random_state=random_state,
            max_iter=200,
            n_init=10,
            init_params="kmeans",
        )
        # Storage
        self.X_train_raw = None
        self.X_test_raw = None
        self.X_train_scl = None
        self.X_test_scl = None
        self.features = None

    # ---- Data preparation ------------------------------------------------
    def prepare(self, df, feature_cols, test_size=0.2):
        self.features = feature_cols
        X = df[feature_cols].dropna().values
        self.X_train_raw, self.X_test_raw = train_test_split(
            X, test_size=test_size, random_state=self.random_state
        )
        self.X_train_scl = self.scaler.fit_transform(self.X_train_raw)
        self.X_test_scl = self.scaler.transform(self.X_test_raw)
        return self

    # ---- EM fitting ------------------------------------------------------
    def fit(self):
        self.model.fit(self.X_train_scl)
        print("=" * 60)
        print("EM FITTING RESULTS")
        print("=" * 60)
        print(f"Converged      : {self.model.converged_}")
        print(f"Iterations     : {self.model.n_iter_}")
        print(f"Train samples  : {len(self.X_train_scl)}")
        print(f"Test samples   : {len(self.X_test_scl)}")
        print(f"Train log-lik  : {self.model.score(self.X_train_scl):.4f}")
        print("=" * 60)

        centers = self.scaler.inverse_transform(self.model.means_)
        print("\nLearned cluster centers (original scale):")
        for k in range(self.n_components):
            print(f"  μ_{k}: {self.features[0]}={centers[k,0]:.2f}, "
                  f"{self.features[1]}={centers[k,1]:.2f}")
        print(f"  Mixture weights: {self.model.weights_.round(4)}")
        print("=" * 60)
        return self

    # ---- Out-of-sample validation ---------------------------------------
    def evaluate(self):
        avg_ll = self.model.score(self.X_test_scl)
        print(f"\nOut-of-Sample Performance:")
        print(f"Average log-likelihood on test set: {avg_ll:.4f}")
        return avg_ll

    # ---- Helper: responsibility surface over original-space grid ----------
    def _responsibility_surface(self, n_grid=200):
        """
        Build a fine grid in ORIGINAL feature space, scale it,
        and compute max posterior responsibility at every pixel.
        """
        pad = 0.05
        x_min = self.X_train_raw[:, 0].min() * (1 - pad)
        x_max = self.X_train_raw[:, 0].max() * (1 + pad)
        y_min = self.X_train_raw[:, 1].min() * (1 - pad)
        y_max = self.X_train_raw[:, 1].max() * (1 + pad)

        xx, yy = np.meshgrid(
            np.linspace(x_min, x_max, n_grid),
            np.linspace(y_min, y_max, n_grid)
        )
        grid_orig = np.c_[xx.ravel(), yy.ravel()]
        grid_scl = self.scaler.transform(grid_orig)

        resp = self.model.predict_proba(grid_scl)          # (n_grid^2, K)
        max_resp = resp.max(axis=1).reshape(xx.shape)    # confidence map

        return xx, yy, max_resp, x_min, x_max, y_min, y_max

    # ---- FIGURE 1: Empirical 2D Density Heatmap -------------------------
    def plot_density_heatmap(self):
        x = self.X_train_raw[:, 0]
        y = self.X_train_raw[:, 1]

        fig = make_subplots(
            rows=2, cols=2,
            specs=[[{"type": "xy"}, {"type": "xy"}],
                   [{"type": "xy", "colspan": 2}, None]],
            subplot_titles=(f"Marginal: {self.features[0]}",
                           f"Marginal: {self.features[1]}",
                           "2D Density Heatmap (Training Data)"),
            row_heights=[0.28, 0.72],
            column_widths=[0.5, 0.5]
        )

        fig.add_trace(go.Histogram(x=x, nbinsx=50,
                                   marker_color="steelblue", opacity=0.7),
                      row=1, col=1)
        fig.add_trace(go.Histogram(x=y, nbinsx=50,
                                   marker_color="coral", opacity=0.7),
                      row=1, col=2)

        fig.add_trace(
            go.Histogram2dContour(
                x=x, y=y, colorscale="Viridis", ncontours=30,
                contours=dict(coloring="heatmap"),
                colorbar=dict(title="Density", x=1.02)
            ), row=2, col=1
        )
        fig.add_trace(
            go.Scatter(x=x, y=y, mode="markers",
                       marker=dict(size=3, color="white", opacity=0.3)),
            row=2, col=1
        )

        fig.update_xaxes(title_text=self.features[0], row=2, col=1)
        fig.update_yaxes(title_text=self.features[1], row=2, col=1)
        fig.update_layout(height=850, width=950, showlegend=False,
                          title_text="Empirical 2D Density with Marginal Distributions")
        fig.show()
        return fig

    # ---- FIGURE 2: Training Assignment Plot ----------------------------
    def plot_training_assignments(self, n_grid=150):
        xx, yy, max_resp, x_min, x_max, y_min, y_max = self._responsibility_surface(n_grid)
        hard_labels = self.model.predict(self.X_train_scl)
        colors = ["#E74C3C", "#2ECC71", "#3498DB"]

        fig = go.Figure()

        # Continuous background: max posterior responsibility
        fig.add_trace(
            go.Contour(
                z=max_resp, x=xx[0, :], y=yy[:, 0],
                colorscale="RdYlBu",
                contours=dict(start=0, end=1, size=0.05),
                colorbar=dict(title="max γ_ik", x=1.02),
                opacity=0.6, name="Confidence"
            )
        )

        # Training points colored by hard argmax
        for k in range(self.n_components):
            mask = hard_labels == k
            fig.add_trace(
                go.Scatter(
                    x=self.X_train_raw[mask, 0], y=self.X_train_raw[mask, 1],
                    mode="markers", name=f"Train Cluster {k}",
                    marker=dict(size=5, color=colors[k],
                               line=dict(width=1, color="black"))
                )
            )

        # Cluster centers μ_k
        centers = self.scaler.inverse_transform(self.model.means_)
        fig.add_trace(
            go.Scatter(
                x=centers[:, 0], y=centers[:, 1],
                mode="markers+text",
                text=[f"μ_{k}" for k in range(self.n_components)],
                textposition="top center",
                marker=dict(size=16, color="gold", symbol="star",
                           line=dict(width=2, color="black")),
                name="Centers"
            )
        )

        fig.update_xaxes(title_text=self.features[0], range=[x_min, x_max])
        fig.update_yaxes(title_text=self.features[1], range=[y_min, y_max])
        fig.update_layout(
            title="Training Assignment: Soft Responsibility Contours with Hard Labels",
            height=650, width=800
        )
        fig.show()
        return fig

    # ---- FIGURE 3: Test Assignment Plot --------------------------------
    def plot_test_assignments(self, n_grid=150):
        xx, yy, max_resp, x_min, x_max, y_min, y_max = self._responsibility_surface(n_grid)
        hard_labels = self.model.predict(self.X_test_scl)
        colors = ["#E74C3C", "#2ECC71", "#3498DB"]

        fig = go.Figure()

        fig.add_trace(
            go.Contour(
                z=max_resp, x=xx[0, :], y=yy[:, 0],
                colorscale="RdYlBu",
                contours=dict(start=0, end=1, size=0.05),
                colorbar=dict(title="max γ_ik", x=1.02),
                opacity=0.5, name="Confidence"
            )
        )

        for k in range(self.n_components):
            mask = hard_labels == k
            if mask.sum() > 0:
                fig.add_trace(
                    go.Scatter(
                        x=self.X_test_raw[mask, 0], y=self.X_test_raw[mask, 1],
                        mode="markers", name=f"Test Cluster {k}",
                        marker=dict(size=9, color=colors[k], symbol="diamond",
                                   line=dict(width=2, color="black"), opacity=0.9)
                    )
                )

        centers = self.scaler.inverse_transform(self.model.means_)
        fig.add_trace(
            go.Scatter(
                x=centers[:, 0], y=centers[:, 1],
                mode="markers+text",
                text=[f"μ_{k}" for k in range(self.n_components)],
                textposition="top center",
                marker=dict(size=16, color="gold", symbol="star",
                           line=dict(width=2, color="black")),
                name="Centers"
            )
        )

        fig.update_xaxes(title_text=self.features[0], range=[x_min, x_max])
        fig.update_yaxes(title_text=self.features[1], range=[y_min, y_max])
        fig.update_layout(
            title="Test Assignment: Out-of-Sample Points on Responsibility Contours",
            height=650, width=800
        )
        fig.show()
        return fig


# ------------------------------------------------------------------
# 3. EXECUTION BLOCK
# ------------------------------------------------------------------
if __name__ == "__main__":
    df = load_credit_data()

    segmenter = GMMFinancialSegmenter(n_components=3, random_state=42)
    segmenter.prepare(df, feature_cols=["PURCHASES", "CREDIT_LIMIT"], test_size=0.2)
    segmenter.fit()
    segmenter.evaluate()

    # Three interactive Plotly figures
    segmenter.plot_density_heatmap()
    segmenter.plot_training_assignments(n_grid=150)
    segmenter.plot_test_assignments(n_grid=150)

100%|██████████| 340k/340k [00:00<00:00, 612kB/s]

Extracting files...


Loaded real Kaggle data: 8949 rows
EM FITTING RESULTS
Converged      : True
Iterations     : 20
Train samples  : 7159
Test samples   : 1790
Train log-lik  : -1.6062

Learned cluster centers (original scale):
  μ_0: PURCHASES=4631.64, CREDIT_LIMIT=9578.34
  μ_1: PURCHASES=183.63, CREDIT_LIMIT=2070.21
  μ_2: PURCHASES=918.78, CREDIT_LIMIT=5744.38
  Mixture weights: [0.1041 0.4428 0.4531]

Out-of-Sample Performance:
Average log-likelihood on test set: -1.6888
